# 03 – Vektorisierung

Hier werden die Wortlisten aus der Vorverarbeitung in Zahlen umgewandelt, damit die Themenmodelle damit rechnen können.

**Eingabe:** 
`data/tokens_final.pkl` (Beschwerden als Wortlisten)

**Ausgabe:** 
zwei Matrizen (Bag-of-Words und TF-IDF) und gensim Wörterbuch

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.corpora import Dictionary


In [3]:
# Hole die gespeicherte Datei aus 02
path = Path("data/tokens_final.pkl")
if not path.exists():
    raise FileNotFoundError("data/tokens_final.pkl fehlt, bitte zuerst 02_vorverarbeitung ausführen.")

df = pd.read_pickle(path)
print(df.shape)
print(df.columns.tolist())

(19807, 6)
['Complaint ID', 'Product', 'Issue', 'text_clean', 'tokens_filtered', 'n_tokens']


In [4]:
print(type(df["tokens_filtered"].iloc[0]))     # sollte <class 'list'> sein
print(df["tokens_filtered"].iloc[0])           # Beispiel einer Wortliste
print(df["n_tokens"].describe())

<class 'list'>
['live', 'city', 'offer', 'tax', 'incentive', 'homeowner', 'certain', 'income', 'qualify', 'homeowner', 'credit', 'tax', 'credit', 'special', 'credit', 'income', 'impound', 'account', 'mortgage', 'company', 'cover', 'tax', 'mortgage', 'company', 'understand', 'program', 'work', 'apply', 'money', 'account', 'tax', 'pay', 'exception', 'mortgage', 'company', 'continue', 'pay', 'money', 'impound', 'account', 'year', 'fall', 'payment', 'unemployment', 'mortgage', 'company', 'begin', 'foreclosure', 'proceeding', 'tell', 'money', 'impound', 'account', 'tax', 'pay', 'catch', 'payment', 'realize', 'tax', 'pay', 'question', 'representative', 'rude', 'tell', 'contact', 'agency', 'begin', 'cooperate', 'tax', 'pay', 'overage', 'account', 'ask', 'receive', 'tell', 'time', 'analysis', 'account', 'current', 'pay', 'time', 'account', 'current', 'pay', 'representative', 'indicate', 'time', 'analysis', 'current', 'hold', 'tax', 'little', 'tax', 'credit', 'pay', 'balance', 'mortgage', 'comp


## Festlegungen
Die Wortlisten aus der Vorverarbeitung werden unverändert übernommen. Scikit-learn nimmt keine eigene Zerlegung, Kleinschreibung oder Filterung mehr vor.

Bag-of-Words und TF-IDF nutzen damit dasselbe Vokabular (3.571 Wörter) und dieselben 19.807 Beschwerden. Sie unterscheiden sich nur in der Gewichtung der Einträge.

In [ ]:
# bag of words test
mini = [
    ["credit", "report", "error"],
    ["credit", "card", "fee", "fee"],
    ["report", "error", "error"],
]

cv_mini = CountVectorizer(analyzer=lambda tokens: tokens)   # Wortlisten unverändert übernehmen
X_mini = cv_mini.fit_transform(mini)

pd.DataFrame(X_mini.toarray(), columns=cv_mini.get_feature_names_out())

,card,credit,error,fee,report
0,0,1,1,0,1
1,1,1,0,2,0
2,0,0,2,0,1


In [6]:
# Erzeugung im scikit-learn Format
cv = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow = cv.fit_transform(df["tokens_filtered"])

print("Form der Matrix:", X_bow.shape)
print("Vokabular:", len(cv.get_feature_names_out()))

Form der Matrix: (19807, 3571)
Vokabular: 3571


In [7]:
n_zellen = X_bow.shape[0] * X_bow.shape[1]
print("Einträge ungleich 0:", X_bow.nnz)
print(f"Anteil der Nullen: {1 - X_bow.nnz / n_zellen:.1%}")

Einträge ungleich 0: 789760
Anteil der Nullen: 98.9%


In [8]:
# Matrix 19.807 × 3.571, 98,9 % Nullen
print(X_bow.shape, X_bow.sum())

(19807, 3571) 1240778


In [9]:
# TF-IDF Test
tv_mini = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf_mini = tv_mini.fit_transform(mini)

pd.DataFrame(X_tfidf_mini.toarray(), columns=tv_mini.get_feature_names_out()).round(2)

,card,credit,error,fee,report
0,0.00,0.58,0.58,0.00,0.58
1,0.42,0.32,0.00,0.85,0.00
2,0.00,0.00,0.89,0.00,0.45


In [10]:
tv = TfidfVectorizer(analyzer=lambda tokens: tokens)
X_tfidf = tv.fit_transform(df["tokens_filtered"])

print("Form der Matrix:", X_tfidf.shape)
print("Vokabular:", len(tv.get_feature_names_out()))
print("Einträge ungleich 0:", X_tfidf.nnz)

Form der Matrix: (19807, 3571)
Vokabular: 3571
Einträge ungleich 0: 789760


In [11]:
words = cv.get_feature_names_out()
assert (words == tv.get_feature_names_out()).all()   # gleiche Reihenfolge in beiden Matrizen

bow_sum = np.asarray(X_bow.sum(axis=0)).ravel()       # Gesamthäufigkeit je Wort
tfidf_sum = np.asarray(X_tfidf.sum(axis=0)).ravel()   # Summe der TF-IDF-Gewichte je Wort

top_bow = pd.Series(bow_sum, index=words).nlargest(15)
top_tfidf = pd.Series(tfidf_sum, index=words).nlargest(15)


top = pd.DataFrame({
    "BoW (Häufigkeit)": top_bow.index,
    "TF-IDF (Summe Gewichte)": top_tfidf.index,
})
top.index = top.index + 1
top.index.name = "Rang"
top

,BoW (Häufigkeit),TF-IDF (Summe Gewichte)
Rang,,
1,account,account
2,credit,credit
3,payment,report
4,report,payment
5,pay,debt
6,loan,pay
7,receive,loan
8,tell,information
9,time,receive


In [12]:
i = 0   
bow_row = pd.Series(X_bow[i].toarray().ravel(), index=words).nlargest(8)
tfidf_row = pd.Series(X_tfidf[i].toarray().ravel(), index=words).nlargest(8)

pd.DataFrame({
    "BoW": [f"{w} ({int(v)})" for w, v in bow_row.items()],
    "TF-IDF": [f"{w} ({v:.2f})" for w, v in tfidf_row.items()],
})

,BoW,TF-IDF
0,tax (12),tax (0.48)
1,account (11),impound (0.30)
2,pay (11),current (0.29)
3,tell (9),pay (0.21)
4,current (8),overage (0.21)
5,credit (7),tell (0.19)
6,mortgage (6),account (0.18)
7,company (5),mortgage (0.18)


## Vergleich: Bag-of-Words und TF-IDF

**Über alle Beschwerden:**
 Die 15 wichtigsten Wörter stimmen in 14 von 15 Fällen überein (nur „day" bzw. „letter" weichen ab), „account" und „credit" stehen bei beiden vorn. Bei TF-IDF steigen thematische Wörter wie „debt" (Platz 12 → 5) und „company" (13 → 10), allgemeine Wörter wie „time" (9 → 14) und „tell" (8 → 11) fallen zurück. (Beispielwerte beziehen sich auf den den Lauf mit der Stichprobe von 20K Beschwerden)
 
 Der Unterschied ist insgesamt gering, weil seltene Wörter und Stoppwörter bereits in der Vorverarbeitung entfernt wurden.

**In einer einzelnen Beschwerde** 
ist der Effekt deutlicher: In Beschwerde 0 (Grundsteuer und Treuhandkonto) hebt TF-IDF die typischen Wörter „impound" und „overage" hervor, während bei Bag-of-Words allgemeine Wörter wie „account" und „pay" vorn stehen.

In [13]:
# Für LDA (gensim) werden dieselben Zählungen im gensim-Format erzeugt
dictionary = Dictionary(df["tokens_filtered"])
corpus = [dictionary.doc2bow(toks) for toks in df["tokens_filtered"]]

print("Wörterbuch:", len(dictionary))                                   # sollte 3571 sein
print("Summe der Zählungen:", sum(c for doc in corpus for _, c in doc))  # muss X_bow.sum() entsprechen

Wörterbuch: 3571
Summe der Zählungen: 1240778


## Zuordnung der Vektorisierung zu den Themenmodellen

- **LDA (gensim) mit Bag-of-Words:** 
LDA ist ein Wahrscheinlichkeitsmodell für gezählte Wörter und erwartet deshalb Häufigkeiten als Eingabe.
- **NMF (scikit-learn) mit TF-IDF:** 
NMF zerlegt die Dokument-Wort-Matrix in kleinere Matrizen. Die Gewichtung durch TF-IDF dampft, dass allgemeine Wörter die Themen dominieren. (Anmerkung: Die Annahme, dass es dadurch ganz verhindert wird, wurde in den Ergebnissen von 06 widerlegt)

Damit bekommt jedes Verfahren die für sie übliche Eingabe, und beide Vektorisierungen kommen zum Einsatz.

**Speicherung:** 
Die Matrizen und das Wörterbuch werden bewusst nicht gespeichert. Sie lassen sich aus `data/tokens_final.pkl` in wenigen Sekunden neu berechnen, und so können sie nicht von der Vorverarbeitung abweichen. Das Notebook `04_themenmodellierung.ipynb` baut die Vektorisierung mit denselben Einstellungen neu auf und prüft sie mit denselben Kennzahlen wie hier .